# System Dependencies

To get started with Unstructured.io, we need a few system-wide dependencies: 

## Poppler (poppler-utils)
Handles PDF processing. It's a library that can extract text, images, and metadata from PDFs. Unstructured uses it to parse PDF documents and convert them into processable text.

## Tesseract (tesseract-ocr) 
Optical Character Recognition (OCR) engine. When you have scanned documents, images with text, or PDFs that are essentially pictures, Tesseract reads the text from these images and converts it to machine-readable text.

## libmagic
File type detection library. It identifies what type of file you're dealing with (PDF, Word doc, image, etc.) by analyzing the file's content, not just the extension. This helps Unstructured choose the right processing method for each document.

In [2]:

import os
import json
from typing import List

# Unstructured for document parsing
from unstructured.partition.pdf import partition_pdf
from unstructured.chunking.title import chunk_by_title

# LangChain components
from langchain_core.documents import Document
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_core.messages import HumanMessage
from dotenv import load_dotenv

load_dotenv()

True

In [3]:
def partition_document(file_path: str):
    """Extract elements from PDF using unstructured"""
    print(f"📄 Partitioning document: {file_path}")
    
    elements = partition_pdf(
        filename=file_path,  # Path to your PDF file
        strategy="hi_res", # Use the most accurate (but slower) processing method of extraction
        infer_table_structure=True, # Keep tables as structured HTML, not jumbled text
        extract_image_block_types=["Image"], # Grab images found in the PDF
        extract_image_block_to_payload=True # Store images as base64 data you can actually use
    )
    
    print(f"✅ Extracted {len(elements)} elements")
    return elements

# Test with your PDF file
file_path = r"C:\RAG_for_EQ\docs\iitk_UTTARKASHI.pdf"  # Change this to your PDF path
elements = partition_document(file_path)

No languages specified, defaulting to English.


📄 Partitioning document: C:\RAG_for_EQ\docs\iitk_UTTARKASHI.pdf


Loading weights: 100%|██████████| 367/367 [00:00<00:00, 1106.51it/s]


✅ Extracted 86 elements


In [4]:
# Gather all images
images = [element for element in elements if element.category == 'Image']
print(f"Found {len(images)} images")

images[0].to_dict()

# Use https://codebeautify.org/base64-to-image-converter to view the base64 text

Found 5 images


{'type': 'Image',
 'element_id': '578034c0f4eee63074f0cf52519d2c8c',
 'text': '2: EPITENTRE UTTARKASHI (8016 km°) CHAMOLI DEHRADOON (9125 km2) (3088 km?) Li. NS" S TEHRI (4421 km?) UTTAR PRADESH',
 'metadata': {'detection_class_prob': 0.8406245112419128,
  'coordinates': {'points': ((np.float64(295.28228759765625),
     np.float64(2561.919921875)),
    (np.float64(295.28228759765625), np.float64(2934.514404296875)),
    (np.float64(1146.19384765625), np.float64(2934.514404296875)),
    (np.float64(1146.19384765625), np.float64(2561.919921875))),
   'system': 'PixelSpace',
   'layout_width': 2477,
   'layout_height': 3550},
  'last_modified': '2026-06-18T13:55:34',
  'filetype': 'application/pdf',
  'languages': ['eng'],
  'page_number': 1,
  'image_base64': '...',
  'image_mime_type': 'image/jpeg',
  'file_directory': 'C:\\RAG_for_EQ\\docs',
  'filename': 'iitk_UTTARKASHI.pdf'}}

In [5]:
# Gather all table
tables = [element for element in elements if element.category == 'Table']
print(f"Found {len(tables)} tables")

tables[0].to_dict()

# Use https://jsfiddle.net/ to view the table html 


Found 0 tables


IndexError: list index out of range

In [6]:
def create_chunks_by_title(elements):
    """Create intelligent chunks using title-based strategy"""
    print("🔨 Creating smart chunks...")
    
    chunks = chunk_by_title(
        elements, # The parsed PDF elements from previous step
        max_characters=3000, # Hard limit - never exceed 3000 characters per chunk
        new_after_n_chars=2400, # Try to start a new chunk after 2400 characters
        combine_text_under_n_chars=500 # Merge tiny chunks under 500 chars with neighbors
    )
    
    print(f"✅ Created {len(chunks)} chunks")
    return chunks

# Create chunks
chunks = create_chunks_by_title(elements)

🔨 Creating smart chunks...
✅ Created 14 chunks


In [7]:
def separate_content_types(chunk):
    """Analyze what types of content are in a chunk"""
    content_data = {
        'text': chunk.text,
        'tables': [],
        'images': [],
        'types': ['text']
    }
    
    # Check for tables and images in original elements
    if hasattr(chunk, 'metadata') and hasattr(chunk.metadata, 'orig_elements'):
        for element in chunk.metadata.orig_elements:
            element_type = type(element).__name__
            
            # Handle tables
            if element_type == 'Table':
                content_data['types'].append('table')
                table_html = getattr(element.metadata, 'text_as_html', element.text)
                content_data['tables'].append(table_html)
            
            # Handle images
            elif element_type == 'Image':
                if hasattr(element, 'metadata') and hasattr(element.metadata, 'image_base64'):
                    content_data['types'].append('image')
                    content_data['images'].append(element.metadata.image_base64)
    
    content_data['types'] = list(set(content_data['types']))
    return content_data

def create_ai_enhanced_summary(text: str, tables: List[str], images: List[str]) -> str:
    """Create AI-enhanced summary for mixed content"""
    
    try:
        # Initialize LLM (needs vision model for images)
        llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash",temperature=0)
        
        # Build the text prompt
        prompt_text = f"""You are creating a searchable description for document content retrieval.

        CONTENT TO ANALYZE:
        TEXT CONTENT:
        {text}

        """
        
        # Add tables if present
        if tables:
            prompt_text += "TABLES:\n"
            for i, table in enumerate(tables):
                prompt_text += f"Table {i+1}:\n{table}\n\n"
        
                prompt_text += """
                YOUR TASK:
                Generate a comprehensive, searchable description that covers:

                1. Key facts, numbers, and data points from text and tables
                2. Main topics and concepts discussed  
                3. Questions this content could answer
                4. Visual content analysis (charts, diagrams, patterns in images)
                5. Alternative search terms users might use

                Make it detailed and searchable - prioritize findability over brevity.

                SEARCHABLE DESCRIPTION:"""

        # Build message content starting with text
        message_content = [{"type": "text", "text": prompt_text}]
        
        # Add images to the message
        for image_base64 in images:
            message_content.append({
                "type": "image_url",
                "image_url": {"url": f"data:image/jpeg;base64,{image_base64}"}
            })
        
        # Send to AI and get response
        message = HumanMessage(content=message_content)
        response = llm.invoke([message])
        
        return response.content
        
    except Exception as e:
        print(f"     ❌ AI summary failed: {e}")
        # Fallback to simple summary
        summary = f"{text[:300]}..."
        if tables:
            summary += f" [Contains {len(tables)} table(s)]"
        if images:
            summary += f" [Contains {len(images)} image(s)]"
        return summary

def summarise_chunks(chunks):
    """Process all chunks with AI Summaries"""
    print("🧠 Processing chunks with AI Summaries...")
    
    langchain_documents = []
    total_chunks = len(chunks)
    
    for i, chunk in enumerate(chunks):
        current_chunk = i + 1
        print(f"   Processing chunk {current_chunk}/{total_chunks}")
        
        # Analyze chunk content
        content_data = separate_content_types(chunk)
        
        # Debug prints
        print(f"     Types found: {content_data['types']}")
        print(f"     Tables: {len(content_data['tables'])}, Images: {len(content_data['images'])}")
        
        # Create AI-enhanced summary if chunk has tables/images
        if content_data['tables'] or content_data['images']:
            print(f"     → Creating AI summary for mixed content...")
            try:
                enhanced_content = create_ai_enhanced_summary(
                    content_data['text'],
                    content_data['tables'], 
                    content_data['images']
                )
                print(f"     → AI summary created successfully")
                print(f"     → Enhanced content preview: {enhanced_content[:200]}...")
            except Exception as e:
                print(f"     ❌ AI summary failed: {e}")
                enhanced_content = content_data['text']
        else:
            print(f"     → Using raw text (no tables/images)")
            enhanced_content = content_data['text']
        
        # Create LangChain Document with rich metadata
        doc = Document(
            page_content=enhanced_content,
            metadata={
                "original_content": json.dumps({
                    "raw_text": content_data['text'],
                    "tables_html": content_data['tables'],
                    "images_base64": content_data['images']
                })
            }
        )
        
        langchain_documents.append(doc)
    
    print(f"✅ Processed {len(langchain_documents)} chunks")
    return langchain_documents


# Process chunks with AI
processed_chunks = summarise_chunks(chunks)

🧠 Processing chunks with AI Summaries...
   Processing chunk 1/14
     Types found: ['text']
     Tables: 0, Images: 0
     → Using raw text (no tables/images)
   Processing chunk 2/14
     Types found: ['text']
     Tables: 0, Images: 0
     → Using raw text (no tables/images)
   Processing chunk 3/14
     Types found: ['text', 'image']
     Tables: 0, Images: 2
     → Creating AI summary for mixed content...
     → AI summary created successfully
     → Enhanced content preview: This document describes the **effects and causes of damage** from an **earthquake** in the **Himalayan region of Uttar Pradesh, India**.

**Key details include:**

*   **Human and Livestock Casualties...
   Processing chunk 4/14
     Types found: ['text']
     Tables: 0, Images: 0
     → Using raw text (no tables/images)
   Processing chunk 5/14
     Types found: ['text']
     Tables: 0, Images: 0
     → Using raw text (no tables/images)
   Processing chunk 6/14
     Types found: ['text']
     Tables: 0, Imag

In [11]:
def export_chunks_to_json(chunks, filename="chunks_export.json"):
    """Export processed chunks to clean JSON format"""
    export_data = []
    
    for i, doc in enumerate(chunks):
        chunk_data = {
            "chunk_id": i + 1,
            "enhanced_content": doc.page_content,
            "metadata": {
                "original_content": json.loads(doc.metadata.get("original_content", "{}"))
            }
        }
        export_data.append(chunk_data)
    
    # Save to file
    with open(filename, 'w', encoding='utf-8') as f:
        json.dump(export_data, f, indent=2, ensure_ascii=False)
    
    print(f"✅ Exported {len(export_data)} chunks to {filename}")
    return export_data

# Export your chunks
json_data = export_chunks_to_json(processed_chunks)

✅ Exported 14 chunks to chunks_export.json


In [8]:
def create_vector_store(documents, persist_directory="dbv1/chroma_db"):
    """Create and persist ChromaDB vector store"""
    print("🔮 Creating embeddings and storing in ChromaDB...")

    embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")   
    
    # Create ChromaDB vector store
    print("--- Creating vector store ---")
    vectorstore = Chroma.from_documents(
        documents=documents,
        embedding=embedding_model,
        persist_directory=persist_directory, 
        collection_metadata={"hnsw:space": "cosine"}
    )
    print("--- Finished creating vector store ---")
    
    print(f"✅ Vector store created and saved to {persist_directory}")
    return vectorstore

# Create the vector store
db = create_vector_store(processed_chunks)

🔮 Creating embeddings and storing in ChromaDB...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2357.49it/s]


--- Creating vector store ---
--- Finished creating vector store ---
✅ Vector store created and saved to dbv1/chroma_db


In [12]:
# After your retrieval
query ="What vulnerability parameters were identified for random rubble stone masonry buildings in the 1991 Uttarkashi earthquake?"
retriever = db.as_retriever(search_kwargs={"k": 5})
chunks = retriever.invoke(query)

# Export to JSON
export_chunks_to_json(chunks, "rag_results.json")

✅ Exported 5 chunks to rag_results.json


[{'chunk_id': 1,
  'enhanced_content': 'Earthquake Engineering, Tenth World Conference © 1994 Balkema, Rotterdam. ISBN 90 5410 060 5\n\nOctober 20, 1991 Uttarkashi (India) earthquake\n\nA.S.Arya CSIR, University of Roorkee, India\n\nABSTRACT: The Uttarkashi earthquake of October 20, 1991 had a Richter magnitude of 6.6, fo- cal depth of 12 km and maximum MM intensity VIII . In the Uttar Pradesh Himalayan hills it caused the death of 768 persons, injured 5066, fully destroyed 20 184 houses and damaged 74 714 more. The main contributor to this scenario was the prevailing type of construction, using field stone either dry packed or built with clay mud. Although the earthquake was not unexpected, code provisions had not been generally implemented. Buildings complying with such provisions underwent no more than minor cracking. Neither the administration nor the population were prepared for the calamity; rescue and relief operations were therefore carried out under very unfavorable conditions

In [9]:
def run_complete_ingestion_pipeline(folder_path: str):
    """Process all PDFs in a folder and create ONE vector database"""

    print("🚀 Starting Multi-PDF RAG Pipeline")
    print("=" * 50)

    all_documents = []

    pdf_files = [
        os.path.join(folder_path, file)
        for file in os.listdir(folder_path)
        if file.lower().endswith(".pdf")
    ]

    print(f"📚 Found {len(pdf_files)} PDF files")

    for pdf_file in pdf_files:

        print(f"\n📄 Processing: {os.path.basename(pdf_file)}")

        elements = partition_document(pdf_file)

        chunks = create_chunks_by_title(elements)

        documents = summarise_chunks(chunks)

        all_documents.extend(documents)

    print(f"\n✅ Total chunks collected: {len(all_documents)}")

    db = create_vector_store(
        all_documents,
        persist_directory="dbv2/chroma_db"
    )

    print("🎉 Multi-PDF Pipeline completed successfully!")

    return db

# Run the complete pipeline

In [10]:
db = run_complete_ingestion_pipeline(
    r"C:\RAG_for_EQ\docs")

No languages specified, defaulting to English.


🚀 Starting Multi-PDF RAG Pipeline
📚 Found 2 PDF files

📄 Processing: iitk_UTTARKASHI.pdf
📄 Partitioning document: C:\RAG_for_EQ\docs\iitk_UTTARKASHI.pdf
✅ Extracted 86 elements
🔨 Creating smart chunks...
✅ Created 14 chunks
🧠 Processing chunks with AI Summaries...
   Processing chunk 1/14
     Types found: ['text']
     Tables: 0, Images: 0
     → Using raw text (no tables/images)
   Processing chunk 2/14
     Types found: ['text']
     Tables: 0, Images: 0
     → Using raw text (no tables/images)
   Processing chunk 3/14
     Types found: ['text', 'image']
     Tables: 0, Images: 2
     → Creating AI summary for mixed content...
     → AI summary created successfully
     → Enhanced content preview: This document describes the **effects and causes of damage** from an **earthquake** in the **Himalayan region of Uttar Pradesh, India**.

**Key details include:**

*   **Human and Livestock Casualties...
   Processing chunk 4/14
     Types found: ['text']
     Tables: 0, Images: 0
     → U

No languages specified, defaulting to English.


✅ Extracted 153 elements
🔨 Creating smart chunks...
✅ Created 24 chunks
🧠 Processing chunks with AI Summaries...
   Processing chunk 1/24
     Types found: ['text', 'image']
     Tables: 0, Images: 1
     → Creating AI summary for mixed content...
     → AI summary created successfully
     → Enhanced content preview: This document and accompanying image depict the devastating aftermath of the **1991 Garhwal earthquake** (magnitude 6.7 Richter Scale) in the **Uttarkashi** region of the **Himalayan hills**, then **U...
   Processing chunk 2/24
     Types found: ['text']
     Tables: 0, Images: 0
     → Using raw text (no tables/images)
   Processing chunk 3/24
     Types found: ['text', 'image']
     Tables: 0, Images: 4
     → Creating AI summary for mixed content...
     → AI summary created successfully
     → Enhanced content preview: This document analyzes the **effects of an earthquake** on **predominant traditional house types**, detailing their **seismic performance** and **dama

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6358.37it/s]


--- Creating vector store ---
--- Finished creating vector store ---
✅ Vector store created and saved to dbv2/chroma_db
🎉 Multi-PDF Pipeline completed successfully!


In [14]:
# Query the vector store
query ="What are the structural vulnerability parameters of random rubble stone masonry buildings in the Uttarkashi earthquake?"

retriever = db.as_retriever(search_kwargs={"k": 5})
chunks = retriever.invoke(query)

def generate_final_answer(chunks, query):
    """Generate final answer using multimodal content"""
    
    try:
        # Initialize LLM (needs vision model for images)
        llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash",temperature=0)
        
        # Build the text prompt
        prompt_text = f"""Based on the following documents, please answer this question: {query}

CONTENT TO ANALYZE:
"""
        
        for i, chunk in enumerate(chunks):
            prompt_text += f"--- Document {i+1} ---\n"
            
            if "original_content" in chunk.metadata:
                original_data = json.loads(chunk.metadata["original_content"])
                
                # Add raw text
                raw_text = original_data.get("raw_text", "")
                if raw_text:
                    prompt_text += f"TEXT:\n{raw_text}\n\n"
                
                # Add tables as HTML
                tables_html = original_data.get("tables_html", [])
                if tables_html:
                    prompt_text += "TABLES:\n"
                    for j, table in enumerate(tables_html):
                        prompt_text += f"Table {j+1}:\n{table}\n\n"
            
            prompt_text += "\n"
        
        prompt_text += f"""
Extract all vulnerability parameters mentioned in the documents.

Output format:

Question:
{query}

Vulnerability Parameters:
1. Parameter A
2. Parameter B
3. Parameter C

Rules:
- Keep the original question at the top.
- Then provide only a numbered list of vulnerability parameters.
- One parameter per line.
- Do not provide explanations.
- Do not create categories.
- Do not write conclusions.

ANSWER:"""

        # Build message content starting with text
        message_content = [{"type": "text", "text": prompt_text}]
        
        # Add all images from all chunks
        for chunk in chunks:
            if "original_content" in chunk.metadata:
                original_data = json.loads(chunk.metadata["original_content"])
                images_base64 = original_data.get("images_base64", [])
                
                for image_base64 in images_base64:
                    message_content.append({
                        "type": "image_url",
                        "image_url": {"url": f"data:image/jpeg;base64,{image_base64}"}
                    })
        
        # Send to AI and get response
        message = HumanMessage(content=message_content)
        response = llm.invoke([message])
        
        return response.content
        
    except Exception as e:
        print(f"❌ Answer generation failed: {e}")
        return "Sorry, I encountered an error while generating the answer."

# Usage
final_answer = generate_final_answer(chunks, query)
print(final_answer)

❌ Answer generation failed: Error calling model 'gemini-2.5-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 37.641552873s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTi